In [ ]:
import sys; sys.path.append('..')
import inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation
from numpy.linalg import norm
import MeshFEM, parallelism, benchmark, utils
import periodic_unit_helper
import numpy.linalg as la

In [ ]:
n_vx = [[0, 0], [0, 1], [0, 2], [0, 3],
        [1, 0], [1, 1], [1, 2], [1, 3]]
n_edge = [(0, 1), (1, 2), (2, 3), 
          (4, 5), (5, 6), (6, 7),
          (0, 4), (1, 5), (2, 6), (3, 7),
          (1, 4), (2, 5), (3, 6)]
triArea = 1

In [ ]:
m, fuseMarkers, fuseSegments = wall_generation.triangulate_channel_walls(n_vx, n_edge, triArea, flags="Y")

In [ ]:
fuseMarkers = [0] * 8

In [ ]:
fuseMarkers[0] = 1

In [ ]:
fuseMarkers[1] = 1

In [ ]:
fuseMarkers[3] = 1
fuseMarkers[4] = 1
fuseMarkers[5] = 1
fuseMarkers[7] = 1

In [ ]:
fuseMarkers

In [ ]:
np.array(fuseMarkers) == 1

In [ ]:
visualization.plot_2d_mesh(m, pointList=np.where(np.array(fuseMarkers) == 1)[0], width=5, height=5)

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, np.array(fuseMarkers) == 1)

In [ ]:
isheet = inflation.InflatableSheet(m, np.array(fuseMarkers) == 1)

In [ ]:
isheet.pressure = 10

In [ ]:
# perturb = np.random.random(isheet.numVars()) * 1e-3

In [ ]:
ipu.periodicVolume()

In [ ]:
# isheet.setVars(isheet.getVars() + perturb)

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
from periodic_simulation_setup import *

In [ ]:
ipu.sheet.setUseTensionFieldEnergy(True)
ipu.sheet.setUseHessianProjectedEnergy(False)
ipu.sheet.disableFusedRegionTensionFieldTheory(False)

ipu.sheet.pressure = 3

In [ ]:
fixedVars, hessianShift = [], 1e-6

In [ ]:
ipu.periodicVolume()

In [ ]:
benchmark.reset()

opts.niter = 100
framerate = 1 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)
benchmark.report()

In [ ]:
np.set_printoptions(precision = 5, suppress = True)

In [ ]:
ipu.getVars().reshape(int(ipu.numVars() / 3), 3)

In [ ]:
import fd_validation

In [ ]:
ipu.sheet.pressure = 100

In [ ]:
ipu.get_use_planar_homogenization()

In [ ]:
fd_validation.hessConvergencePlot(ipu, customArgs = {"energyType": inflation.InflatableSheet.EnergyType.Pressure})

In [ ]:
fd_validation.hessConvergencePlot(ipu, customArgs = {"energyType": inflation.InflatableSheet.EnergyType.Elastic})

In [ ]:
deformed_pos = ipu.sheet.getVars().reshape(int(ipu.sheet.numVars() / 3), 3)

In [ ]:
deformed_pos

In [ ]:
np.cross((deformed_pos[2] - deformed_pos[1]), (deformed_pos[3] - deformed_pos[1])).dot(deformed_pos[1] - deformed_pos[5])

In [ ]:
ipu.periodicVolume()

In [ ]:
ipu.sheet.volume()

### Validate volume

In [ ]:
interior_tris = [[deformed_pos[1], deformed_pos[6], deformed_pos[2]], 
       [deformed_pos[2], deformed_pos[6], deformed_pos[7]],
       [deformed_pos[2], deformed_pos[7], deformed_pos[9]],
       [deformed_pos[2], deformed_pos[9], deformed_pos[4]],
                [deformed_pos[6], deformed_pos[1], deformed_pos[3]], 
       [deformed_pos[6], deformed_pos[3], deformed_pos[8]],
       [deformed_pos[8], deformed_pos[3], deformed_pos[9]],
       [deformed_pos[9], deformed_pos[3], deformed_pos[4]]]

In [ ]:
boundary_tris = [[deformed_pos[1], deformed_pos[2], deformed_pos[3]], 
       [deformed_pos[2], deformed_pos[4], deformed_pos[3]],
       [deformed_pos[9], deformed_pos[7], deformed_pos[8]],
       [deformed_pos[7], deformed_pos[6], deformed_pos[8]]]

In [ ]:
boundary_volume = 0
for tri in boundary_tris:
    boundary_volume += la.det(tri)
    print(la.det(tri))

In [ ]:
boundary_volume / 6

In [ ]:
volume = 0
for tri in interior_tris:
    volume += la.det(tri)
    print(la.det(tri))

In [ ]:
volume / 6

In [ ]:
(boundary_volume + volume) / 6